# LASA member registration model (training notebook)

Designed for **Google Colab**. Trains member-level **logistic regression** on `LASA2018 - LASA2026.xlsx` and sums predicted probabilities into expected paid registrations per congress.

**Colab quick start**
1. Open this notebook in Colab.
2. Upload `LASA2018 - LASA2026.xlsx` with the **Files** panel (left sidebar → folder icon → upload), **or** run the setup cell and pick the file when prompted.
3. **Runtime → Run all**.

Do not publish the member workbook in a public repository.

**Label:** confirm with LASA whether the outcome is `Registered == 1` or paid only (`Reg. Amount > 0`). Paris **2026** is excluded from training by default.

In [ ]:
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

XLSX_NAME = "LASA2018 - LASA2026.xlsx"

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "openpyxl", "joblib"],
        check=True,
    )
    WORK_DIR = Path("/content")
    XLSX_PATH = WORK_DIR / XLSX_NAME
    ARTIFACT_DIR = WORK_DIR / "lasa_artifacts"
    if not XLSX_PATH.is_file():
        from google.colab import files  # type: ignore

        print(
            f"No {XLSX_NAME} in /content yet. "
            "Upload it in the Files panel first, or choose it below."
        )
        uploaded = files.upload()
        if XLSX_NAME not in uploaded:
            raise FileNotFoundError(f"Expected uploaded file named {XLSX_NAME!r}")
        XLSX_PATH.write_bytes(uploaded[XLSX_NAME])
    REPO_ROOT = None
else:
    REPO_ROOT = None
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / XLSX_NAME).is_file():
            REPO_ROOT = candidate
            break
        if (candidate / "README.md").is_file() and (candidate / "src" / "lasa_pilot").is_dir():
            REPO_ROOT = candidate
            break
    if REPO_ROOT is None:
        raise FileNotFoundError(
            "Could not find repo root. Open this notebook from the LASA project folder."
        )
    XLSX_PATH = REPO_ROOT / XLSX_NAME
    if not XLSX_PATH.is_file():
        raise FileNotFoundError(f"Missing {XLSX_PATH}")
    ARTIFACT_DIR = REPO_ROOT / "src" / "lasa_pilot" / "artifacts"
    sys.path.insert(0, str(REPO_ROOT / "src"))

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("Running in Colab:", IN_COLAB)
print("Excel:", XLSX_PATH, "→", XLSX_PATH.is_file())
print("Artifacts:", ARTIFACT_DIR)
if REPO_ROOT is not None:
    print("Repo root:", REPO_ROOT)

In [ ]:
from __future__ import annotations

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from typing import Callable
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from lasa_pilot.tools import haversine_km
except ImportError:
    import math

    _COORDINATES = {
        "barcelona": (41.39, 2.17),
        "buenos aires": (-34.60, -58.38),
        "rome": (41.90, 12.50),
        "berlin": (52.52, 13.40),
        "guatemala city": (14.63, -90.51),
        "boston": (42.36, -71.06),
        "lima": (-12.05, -77.04),
        "guadalajara": (20.66, -103.35),
        "montreal": (45.50, -73.57),
        "new york": (40.71, -74.01),
        "mexico city": (19.43, -99.13),
        "sao paulo": (-23.55, -46.63),
        "madrid": (40.42, -3.70),
        "vancouver": (49.28, -123.12),
        "bogota": (4.71, -74.07),
        "san francisco": (37.77, -122.42),
        "paris": (48.86, 2.35),
    }

    def _norm_city(city: str) -> str:
        return " ".join(city.strip().lower().split())

    def haversine_km(origin: str, destination: str) -> float:
        lat1, lon1 = _COORDINATES[_norm_city(origin)]
        lat2, lon2 = _COORDINATES[_norm_city(destination)]
        radius = 6371.0
        phi1, phi2 = math.radians(lat1), math.radians(lat2)
        dphi = math.radians(lat2 - lat1)
        dlambda = math.radians(lon2 - lon1)
        a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
        return round(2 * radius * math.asin(math.sqrt(a)), 1)

## Congress metadata

Sheet names in the workbook are the source of host city and format (in-person vs virtual).

In [ ]:
CONGRESS_BY_YEAR: dict[int, dict[str, object]] = {
    2018: {"host_city": "Barcelona", "host_country": "Spain", "in_person": True},
    2019: {"host_city": "Boston", "host_country": "USA", "in_person": True},
    2020: {"host_city": "Guadalajara", "host_country": "Mexico", "in_person": False},
    2021: {"host_city": "Virtual", "host_country": "Virtual", "in_person": False},
    2022: {"host_city": "Virtual", "host_country": "Virtual", "in_person": False},
    2023: {"host_city": "Vancouver", "host_country": "Canada", "in_person": True},
    2024: {"host_city": "Bogota", "host_country": "Colombia", "in_person": True},
    2025: {"host_city": "San Francisco", "host_country": "USA", "in_person": True},
    2026: {"host_city": "Paris", "host_country": "France", "in_person": True},
}

# Representative city for great-circle distance (must exist in lasa_pilot.tools.COORDINATES).
COUNTRY_TO_CITY: dict[str, str] = {
    "UNITED STATES": "New York",
    "CANADA": "Montreal",
    "MEXICO": "Mexico City",
    "BRAZIL": "Sao Paulo",
    "ARGENTINA": "Buenos Aires",
    "CHILE": "Buenos Aires",
    "COLOMBIA": "Mexico City",
    "ECUADOR": "Lima",
    "PERU": "Lima",
    "SPAIN": "Madrid",
    "FRANCE": "Paris",
    "UNITED KINGDOM": "London",
}

DEFAULT_ORIGIN_CITY = "Mexico City"
TRAINING_YEARS = tuple(y for y in sorted(CONGRESS_BY_YEAR) if y != 2026)
USE_PAID_LABEL = True  # False => Registered == 1 only
USD_PER_KM = 0.10

## Load and normalize member rows

In [ ]:
def normalize_country(value: object) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "UNKNOWN"
    return " ".join(str(value).strip().upper().split())


def reg_amount_column(columns: list[str]) -> str:
    for name in columns:
        if "amount" in name.lower():
            return name
    raise ValueError(f"No registration amount column in {columns}")


def load_member_table(path: Path) -> pd.DataFrame:
    xl = pd.ExcelFile(path)
    chunks: list[pd.DataFrame] = []
    for sheet in xl.sheet_names:
        raw = pd.read_excel(path, sheet_name=sheet)
        amt_col = reg_amount_column(list(raw.columns))
        frame = raw.rename(columns={amt_col: "reg_amount"}).copy()
        frame = frame[[c for c in frame.columns if c in {
            "Member Number", "Cong. Year", "Country", "Submitted", "Accepted",
            "Rejected", "Registered", "reg_amount",
        }]]
        frame["sheet_name"] = sheet
        chunks.append(frame)
    out = pd.concat(chunks, ignore_index=True)
    out["year"] = out["Cong. Year"].astype(int)
    out["member_number"] = out["Member Number"].astype(int)
    out["country"] = out["Country"].map(normalize_country)
    out["reg_amount"] = pd.to_numeric(out["reg_amount"], errors="coerce").fillna(0.0)
    out["registered"] = (out["Registered"].fillna(0).astype(int) == 1).astype(int)
    if USE_PAID_LABEL:
        out["label"] = ((out["registered"] == 1) & (out["reg_amount"] > 0)).astype(int)
    else:
        out["label"] = out["registered"]
    for year, meta in CONGRESS_BY_YEAR.items():
        mask = out["year"] == year
        out.loc[mask, "host_city"] = meta["host_city"]
        out.loc[mask, "host_country"] = meta["host_country"]
        out.loc[mask, "in_person"] = bool(meta["in_person"])
    return out


members = load_member_table(XLSX_PATH)
members = members[members["year"].isin(TRAINING_YEARS)].copy()
print(members.shape)
members.groupby("year")["label"].agg(["count", "sum", "mean"]).round(3)

## Features (history uses prior congresses only)

In [ ]:
def origin_city_for_country(country: str) -> str:
    return COUNTRY_TO_CITY.get(country, DEFAULT_ORIGIN_CITY)


def travel_cost_usd(host_city: str, country: str, usd_per_km: float) -> float:
    if host_city == "Virtual":
        return 0.0
    origin = origin_city_for_country(country)
    try:
        km = haversine_km(origin, host_city)
    except KeyError:
        km = haversine_km(DEFAULT_ORIGIN_CITY, host_city)
    return round(km * usd_per_km, 2)


def add_history_features(df: pd.DataFrame) -> pd.DataFrame:
    ordered = df.sort_values(["member_number", "year"]).copy()
    grouped = ordered.groupby("member_number", sort=False)
    ordered["prior_registrations"] = grouped["label"].cumsum() - ordered["label"]
    ordered["prior_submissions"] = grouped["Submitted"].cumsum() - ordered["Submitted"]
    ordered["prior_acceptances"] = grouped["Accepted"].cumsum() - ordered["Accepted"]
    ordered["ever_registered_before"] = (ordered["prior_registrations"] > 0).astype(int)
    ordered["travel_cost_usd"] = [
        travel_cost_usd(host, country, USD_PER_KM)
        for host, country in zip(ordered["host_city"], ordered["country"], strict=True)
    ]
    ordered["in_person_int"] = ordered["in_person"].astype(int)
    return ordered


model_df = add_history_features(members)
FEATURE_COLUMNS = [
    "country",
    "host_city",
    "host_country",
    "in_person_int",
    "prior_registrations",
    "prior_submissions",
    "prior_acceptances",
    "ever_registered_before",
    "travel_cost_usd",
]
TARGET = "label"
model_df[FEATURE_COLUMNS].head()

## Model pipelines (logistic regression vs random forest)

In [ ]:
CATEGORICAL = ["country", "host_city", "host_country"]
NUMERIC = [
    "in_person_int",
    "prior_registrations",
    "prior_submissions",
    "prior_acceptances",
    "ever_registered_before",
    "travel_cost_usd",
]


def make_preprocessor(*, scale_numeric: bool) -> ColumnTransformer:
    numeric_step = StandardScaler() if scale_numeric else "passthrough"
    return ColumnTransformer(
        transformers=[
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore", min_frequency=25),
                CATEGORICAL,
            ),
            ("num", numeric_step, NUMERIC),
        ]
    )


def make_logreg_pipeline() -> Pipeline:
    clf = LogisticRegression(
        penalty="l2",
        C=1.0,
        class_weight="balanced",
        max_iter=5000,
        random_state=42,
    )
    return Pipeline([("preprocess", make_preprocessor(scale_numeric=True)), ("clf", clf)])


def make_rf_pipeline() -> Pipeline:
    clf = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    return Pipeline([("preprocess", make_preprocessor(scale_numeric=False)), ("clf", clf)])


def congress_totals(frame: pd.DataFrame) -> pd.Series:
    return frame.groupby("year")[TARGET].sum().astype(int)


def predict_congress_total(pipe: Pipeline, frame: pd.DataFrame) -> pd.Series:
    probs = pipe.predict_proba(frame[FEATURE_COLUMNS])[:, 1]
    scored = frame[["year"]].copy()
    scored["pred"] = probs
    return scored.groupby("year")["pred"].sum()


def run_loocv(pipe_factory: Callable[[], Pipeline], model_name: str) -> pd.DataFrame:
    """Leave-one-congress-out evaluation for one model type."""
    actual_totals = congress_totals(model_df)
    fold_rows: list[dict[str, object]] = []
    for holdout_year in sorted(model_df["year"].unique()):
        train = model_df[model_df["year"] != holdout_year]
        test = model_df[model_df["year"] == holdout_year]
        pipe = pipe_factory()
        pipe.fit(train[FEATURE_COLUMNS], train[TARGET])
        pred_total = float(predict_congress_total(pipe, test)[holdout_year])
        actual_total = int(actual_totals[holdout_year])
        member_auc = roc_auc_score(
            test[TARGET], pipe.predict_proba(test[FEATURE_COLUMNS])[:, 1]
        )
        fold_rows.append(
            {
                "model": model_name,
                "holdout_year": holdout_year,
                "host_city": CONGRESS_BY_YEAR[holdout_year]["host_city"],
                "actual_paid": actual_total,
                "predicted_paid": round(pred_total, 1),
                "abs_error": round(abs(pred_total - actual_total), 1),
                "member_auc": round(member_auc, 3),
            }
        )
    report = pd.DataFrame(fold_rows)
    report["pct_error"] = (
        100 * report["abs_error"] / report["actual_paid"].clip(lower=1)
    ).round(1)
    return report

## Leave-one-congress-out evaluation (compare models)

In [ ]:
actual_totals = congress_totals(model_df)

print("Running LOOCV: logistic regression...")
cv_report_logreg = run_loocv(make_logreg_pipeline, "logistic_regression")
print("Running LOOCV: random forest (may take a few minutes)...")
cv_report_rf = run_loocv(make_rf_pipeline, "random_forest")

cv_report = pd.concat([cv_report_logreg, cv_report_rf], ignore_index=True)

model_summary = (
    cv_report.groupby("model", as_index=False)
    .agg(
        congress_mae=("abs_error", "mean"),
        mean_member_auc=("member_auc", "mean"),
        mean_pct_error=("pct_error", "mean"),
    )
    .round(3)
    .sort_values("congress_mae")
)
best_model_name = model_summary.iloc[0]["model"]
print("\nSummary (lower congress_mae is better for attendance totals):")
print(model_summary.to_string(index=False))
print(f"Best on LOOCV congress MAE: {best_model_name}")
cv_report

## Member-level accuracy, precision, and recall (LOOCV)

Uses **out-of-fold** predictions (one held-out congress per fold). Class labels use the same target as training (`label`). Predictions use **probability ≥ 0.5** as registered (paid). Registration is imbalanced, so read these together with **congress MAE** above.

In [ ]:
from typing import Callable

from sklearn.metrics import accuracy_score, precision_score, recall_score

CLASSIFICATION_THRESHOLD = 0.5


def member_metrics_oof(
    pipe_factory: Callable[[], Pipeline],
    model_name: str,
    threshold: float = CLASSIFICATION_THRESHOLD,
) -> tuple[pd.DataFrame, dict[str, float]]:
    fold_rows: list[dict[str, object]] = []
    y_all: list[int] = []
    pred_all: list[int] = []

    for holdout_year in sorted(model_df["year"].unique()):
        train = model_df[model_df["year"] != holdout_year]
        test = model_df[model_df["year"] == holdout_year]
        pipe = pipe_factory()
        pipe.fit(train[FEATURE_COLUMNS], train[TARGET])
        probs = pipe.predict_proba(test[FEATURE_COLUMNS])[:, 1]
        y_true = test[TARGET].to_numpy()
        y_pred = (probs >= threshold).astype(int)

        fold_rows.append(
            {
                "model": model_name,
                "holdout_year": holdout_year,
                "host_city": CONGRESS_BY_YEAR[holdout_year]["host_city"],
                "accuracy": round(accuracy_score(y_true, y_pred), 3),
                "precision": round(
                    precision_score(y_true, y_pred, zero_division=0), 3
                ),
                "recall": round(recall_score(y_true, y_pred, zero_division=0), 3),
                "positive_rate_actual": round(float(y_true.mean()), 3),
                "positive_rate_predicted": round(float(y_pred.mean()), 3),
            }
        )
        y_all.extend(y_true.tolist())
        pred_all.extend(y_pred.tolist())

    y_arr = np.asarray(y_all, dtype=int)
    pred_arr = np.asarray(pred_all, dtype=int)
    pooled = {
        "accuracy": round(accuracy_score(y_arr, pred_arr), 3),
        "precision": round(precision_score(y_arr, pred_arr, zero_division=0), 3),
        "recall": round(recall_score(y_arr, pred_arr, zero_division=0), 3),
    }
    return pd.DataFrame(fold_rows), pooled


print(
    f"Member-level metrics (LOOCV, threshold={CLASSIFICATION_THRESHOLD})..."
)
fold_logreg, pooled_logreg = member_metrics_oof(
    make_logreg_pipeline, "logistic_regression"
)
fold_rf, pooled_rf = member_metrics_oof(make_rf_pipeline, "random_forest")

member_metrics_by_fold = pd.concat([fold_logreg, fold_rf], ignore_index=True)

member_metrics_summary = pd.DataFrame(
    [
        {
            "model": "logistic_regression",
            "accuracy_pooled_oof": pooled_logreg["accuracy"],
            "precision_pooled_oof": pooled_logreg["precision"],
            "recall_pooled_oof": pooled_logreg["recall"],
            "accuracy_mean_by_fold": round(fold_logreg["accuracy"].mean(), 3),
            "precision_mean_by_fold": round(fold_logreg["precision"].mean(), 3),
            "recall_mean_by_fold": round(fold_logreg["recall"].mean(), 3),
        },
        {
            "model": "random_forest",
            "accuracy_pooled_oof": pooled_rf["accuracy"],
            "precision_pooled_oof": pooled_rf["precision"],
            "recall_pooled_oof": pooled_rf["recall"],
            "accuracy_mean_by_fold": round(fold_rf["accuracy"].mean(), 3),
            "precision_mean_by_fold": round(fold_rf["precision"].mean(), 3),
            "recall_mean_by_fold": round(fold_rf["recall"].mean(), 3),
        },
    ]
)

print("\nSummary (pooled = all OOF rows; mean_by_fold = average across 8 congress folds):")
print(member_metrics_summary.to_string(index=False))
print("\nPer held-out congress:")
member_metrics_by_fold

## Fit final model on all training years and save

In [ ]:
final_logreg = make_logreg_pipeline()
final_rf = make_rf_pipeline()
final_logreg.fit(model_df[FEATURE_COLUMNS], model_df[TARGET])
final_rf.fit(model_df[FEATURE_COLUMNS], model_df[TARGET])

final_pipes = {
    "logistic_regression": final_logreg,
    "random_forest": final_rf,
}
final_pipe = final_pipes[best_model_name]

for name, pipe in final_pipes.items():
    artifact_path = ARTIFACT_DIR / f"member_registration_{name}.joblib"
    joblib.dump(
        {
            "pipeline": pipe,
            "model_name": name,
            "feature_columns": FEATURE_COLUMNS,
            "use_paid_label": USE_PAID_LABEL,
            "training_years": TRAINING_YEARS,
            "usd_per_km": USD_PER_KM,
            "congress_by_year": CONGRESS_BY_YEAR,
            "loocv_summary": model_summary.to_dict(orient="records"),
        },
        artifact_path,
    )
    print("Saved:", artifact_path)

if IN_COLAB:
    from google.colab import files  # type: ignore

    for name in final_pipes:
        files.download(str(ARTIFACT_DIR / f"member_registration_{name}.joblib"))
    print("Downloaded both model files.")

in_sample = pd.DataFrame({"year": actual_totals.index, "actual_paid": actual_totals.values})
in_sample["host_city"] = in_sample["year"].map(lambda y: CONGRESS_BY_YEAR[y]["host_city"])
in_sample["pred_logreg"] = predict_congress_total(final_logreg, model_df).round(1).values
in_sample["pred_rf"] = predict_congress_total(final_rf, model_df).round(1).values
in_sample

## Counterfactual: expected attendance for a candidate host

Uses the **latest member list** (max training year) and swaps host fields + travel cost. This is illustrative until LASA confirms the candidate set and counting rule.

In [ ]:
CANDIDATE_HOSTS = [
    ("Rome", "Italy"),
    ("Berlin", "Germany"),
    ("Buenos Aires", "Argentina"),
    ("Guatemala City", "Guatemala"),
    ("Barcelona", "Spain"),
]

latest_year = max(TRAINING_YEARS)
baseline = model_df[model_df["year"] == latest_year].copy()


def scenario_frame(host_city: str, host_country: str) -> pd.DataFrame:
    scenario = baseline.copy()
    scenario["host_city"] = host_city
    scenario["host_country"] = host_country
    scenario["in_person_int"] = 1
    scenario["travel_cost_usd"] = [
        travel_cost_usd(host_city, country, USD_PER_KM) for country in scenario["country"]
    ]
    return scenario


scenario_rows: list[dict[str, object]] = []
for host_city, host_country in CANDIDATE_HOSTS:
    scen = scenario_frame(host_city, host_country)
    row = {
        "host_city": host_city,
        "host_country": host_country,
        "member_rows_scored": len(scen),
        "reference_year_for_roster": latest_year,
    }
    for name, pipe in final_pipes.items():
        total = float(pipe.predict_proba(scen[FEATURE_COLUMNS])[:, 1].sum())
        row[f"expected_{name}"] = round(total, 1)
    scenario_rows.append(row)

scenario_df = pd.DataFrame(scenario_rows).sort_values(
    f"expected_{best_model_name}", ascending=False
)
scenario_df

## Model evaluation plots

**Leave-one-congress-out (LOOCV)** charts compare **logistic regression** and **random forest**. Lower congress **MAE** is the main metric for attendance totals. Member **ROC** / **calibration** use out-of-fold probabilities.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_curve

MODEL_STYLES = {
    "logistic_regression": {"color": "#2563eb", "label": "Logistic regression"},
    "random_forest": {"color": "#ea580c", "label": "Random forest"},
}


def collect_oof_probs(pipe_factory: Callable[[], Pipeline], model_name: str) -> pd.DataFrame:
    chunks: list[pd.DataFrame] = []
    for holdout_year in sorted(model_df["year"].unique()):
        train = model_df[model_df["year"] != holdout_year]
        test = model_df[model_df["year"] == holdout_year]
        pipe = pipe_factory()
        pipe.fit(train[FEATURE_COLUMNS], train[TARGET])
        probs = pipe.predict_proba(test[FEATURE_COLUMNS])[:, 1]
        chunk = test[["year", TARGET]].copy()
        chunk["prob"] = probs
        chunk["model"] = model_name
        chunks.append(chunk)
    return pd.concat(chunks, ignore_index=True)


print("Building out-of-fold predictions for plots...")
oof = pd.concat(
    [
        collect_oof_probs(make_logreg_pipeline, "logistic_regression"),
        collect_oof_probs(make_rf_pipeline, "random_forest"),
    ],
    ignore_index=True,
)

year_labels = {
    int(row["holdout_year"]): f"{int(row['holdout_year'])}\n({row['host_city']})"
    for _, row in cv_report_logreg.iterrows()
}
years_ordered = sorted(year_labels)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Logistic regression vs random forest (LOOCV)", fontsize=14, y=1.02)

# 1) Actual vs predicted (both models)
ax = axes[0, 0]
all_pred = cv_report["predicted_paid"]
all_actual = cv_report["actual_paid"]
lims = [min(all_actual.min(), all_pred.min()) * 0.9, max(all_actual.max(), all_pred.max()) * 1.05]
ax.plot(lims, lims, "--", color="gray", linewidth=1, label="Perfect prediction")
for model_name, style in MODEL_STYLES.items():
    part = cv_report[cv_report["model"] == model_name]
    ax.scatter(
        part["actual_paid"],
        part["predicted_paid"],
        s=70,
        color=style["color"],
        label=style["label"],
        alpha=0.85,
    )
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Actual paid registrations")
ax.set_ylabel("Predicted paid (sum of probabilities)")
ax.set_title("Congress totals: actual vs predicted")
ax.legend(loc="lower right")

# 2) MAE comparison
ax = axes[0, 1]
mae_vals = [model_summary.set_index("model").loc[m, "congress_mae"] for m in MODEL_STYLES]
bars = ax.bar(
    [MODEL_STYLES[m]["label"] for m in MODEL_STYLES],
    mae_vals,
    color=[MODEL_STYLES[m]["color"] for m in MODEL_STYLES],
)
ax.set_ylabel("Mean absolute error (registrations)")
ax.set_title("LOOCV congress total MAE (lower is better)")
for bar, val in zip(bars, mae_vals, strict=True):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 40, f"{val:.0f}", ha="center", fontsize=10)

# 3) ROC curves
ax = axes[1, 0]
ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1, label="Random")
for model_name, style in MODEL_STYLES.items():
    part = oof[oof["model"] == model_name]
    fpr, tpr, _ = roc_curve(part[TARGET], part["prob"])
    mean_auc = cv_report[cv_report["model"] == model_name]["member_auc"].mean()
    ax.plot(fpr, tpr, color=style["color"], linewidth=2, label=f"{style['label']} (AUC {mean_auc:.2f})")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("Member-level ROC (out-of-fold)")
ax.legend(loc="lower right")

# 4) Calibration
ax = axes[1, 1]
ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1, label="Perfect calibration")
for model_name, style in MODEL_STYLES.items():
    part = oof[oof["model"] == model_name]
    frac_pos, mean_pred = calibration_curve(
        part[TARGET], part["prob"], n_bins=10, strategy="quantile"
    )
    ax.plot(mean_pred, frac_pos, "o-", color=style["color"], label=style["label"])
ax.set_xlabel("Mean predicted probability (bin)")
ax.set_ylabel("Fraction registered (paid)")
ax.set_title("Calibration (member-level, LOOCV)")
ax.legend(loc="upper left")

plt.tight_layout()
eval_fig_path = ARTIFACT_DIR / "model_comparison_loocv.png"
fig.savefig(eval_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", eval_fig_path)

# Per-year predicted totals (both models)
fig2, ax2 = plt.subplots(figsize=(13, 5))
x = np.arange(len(years_ordered))
width = 0.22
actual_by_year = cv_report_logreg.set_index("holdout_year")["actual_paid"]
ax2.bar(x - width, [actual_by_year[y] for y in years_ordered], width, label="Actual", color="#16a34a")
for i, model_name in enumerate(MODEL_STYLES):
    part = cv_report[cv_report["model"] == model_name].set_index("holdout_year")
    offset = width * i
    ax2.bar(
        x + offset,
        [part.loc[y, "predicted_paid"] for y in years_ordered],
        width,
        label=MODEL_STYLES[model_name]["label"],
        color=MODEL_STYLES[model_name]["color"],
    )
ax2.set_xticks(x + width / 2)
ax2.set_xticklabels([year_labels[y] for y in years_ordered], rotation=45, ha="right", fontsize=8)
ax2.set_ylabel("Paid registrations")
ax2.set_title("Held-out year: actual vs predicted totals")
ax2.legend()
plt.tight_layout()
year_fig_path = ARTIFACT_DIR / "model_comparison_by_year.png"
fig2.savefig(year_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", year_fig_path)

# In-sample (both models)
fig3, ax3 = plt.subplots(figsize=(6, 6))
lims3 = [
    min(in_sample["actual_paid"].min(), in_sample[["pred_logreg", "pred_rf"]].min().min()) * 0.9,
    max(in_sample["actual_paid"].max(), in_sample[["pred_logreg", "pred_rf"]].max().max()) * 1.05,
]
ax3.plot(lims3, lims3, "--", color="gray", linewidth=1)
ax3.scatter(in_sample["actual_paid"], in_sample["pred_logreg"], color="#2563eb", label="Logistic regression")
ax3.scatter(in_sample["actual_paid"], in_sample["pred_rf"], color="#ea580c", label="Random forest")
ax3.set_xlim(lims3)
ax3.set_ylim(lims3)
ax3.set_xlabel("Actual paid registrations")
ax3.set_ylabel("Predicted (in-sample)")
ax3.set_title("In-sample fit (optimistic — not for reporting)")
ax3.legend()
plt.tight_layout()
insample_fig_path = ARTIFACT_DIR / "model_comparison_insample.png"
fig3.savefig(insample_fig_path, dpi=150, bbox_inches="tight")
plt.show()

# Candidate scenarios (both models)
fig4, ax4 = plt.subplots(figsize=(9, max(4, 0.55 * len(scenario_df))))
y_pos = np.arange(len(scenario_df))
bar_h = 0.35
ax4.barh(y_pos - bar_h / 2, scenario_df["expected_logistic_regression"], height=bar_h, color="#2563eb", label="Logistic regression")
ax4.barh(y_pos + bar_h / 2, scenario_df["expected_random_forest"], height=bar_h, color="#ea580c", label="Random forest")
ax4.set_yticks(y_pos)
ax4.set_yticklabels(scenario_df["host_city"])
ax4.set_xlabel("Expected paid registrations (counterfactual)")
ax4.set_title(f"Candidate hosts (roster from {latest_year}; illustrative)")
ax4.legend(loc="lower right")
plt.tight_layout()
scenario_fig_path = ARTIFACT_DIR / "candidate_host_comparison.png"
fig4.savefig(scenario_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", scenario_fig_path)

if IN_COLAB:
    from google.colab import files  # type: ignore

    for path in (eval_fig_path, year_fig_path, insample_fig_path, scenario_fig_path):
        files.download(str(path))